# Graph-Enriched Search

This is the core notebook of Lab 4. In the previous notebook, vector search returned isolated chunks — text fragments ranked by similarity with no surrounding context. **Graph-enriched search** changes that by following relationships from each matched chunk to connected entities in the knowledge graph.

After finding semantically similar chunks, the agent traverses graph relationships to pull in document metadata, neighboring chunks, and connected entities like companies, products, and risk factors. The graph turns text retrieval into knowledge retrieval.

**Learning Objectives:**
- Understand how graph traversal enriches vector search results with structured context
- Compare results between pure vector search and graph-enriched search
- Combine vector similarity search with Cypher graph traversal

**How it works:**
1. Custom `@tool` wrappers encapsulate embedding generation and Cypher execution — the LLM never sees raw float arrays
2. Each tool runs a vector search to find matching chunks via MCP
3. For each match, the Cypher query traverses graph relationships to gather additional context:
   - Previous and next chunks (for a wider text window)
   - Source document metadata
   - Connected entities (companies, risk factors, etc.)
4. The enriched results give the LLM more context for generating answers

This mirrors the `VectorCypherRetriever` from the neo4j-graphrag library, but executed through MCP.

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database, embeddings, and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx nest-asyncio -q

## 1. Configuration and Imports

In [ ]:
import os

import nest_asyncio
from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

nest_asyncio.apply()

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("Configuration loaded!")

## 2. Shared Utilities and MCP Connection

The embedding helper from `lib/` centralizes Bedrock embedding configuration. The MCP connection is opened persistently for the notebook session, and the Cypher query tool is discovered automatically.

### Entity Relationships in the Graph

The knowledge graph connects chunks to companies through document ownership:

```
(Chunk)-[:FROM_DOCUMENT]->(Document)<-[:FILED]-(Company)  the company that filed the document
(Product)-[:FROM_CHUNK]->(Chunk)     products mentioned in a chunk
(Company)-[:FACES_RISK]->(RiskFactor)  risks associated with companies
```

By traversing these relationships, the agent can augment each vector match with structured entity data.

In [ ]:
from botocore.config import Config as BotocoreConfig
from lib.lab_4_data_utils import get_embedding

# Verify the shared embedding function works
test_embedding = get_embedding("test")
print(f"Embedding dimensions: {len(test_embedding)}")

# Model
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools and find the Cypher query tool
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")

cypher_tool = next((n for n in tool_names if "read-cypher" in n), None)
assert cypher_tool, f"No Cypher query tool found among: {tool_names}"

print(f"\nModel: {MODEL_ID}")
print(f"Cypher tool: {cypher_tool}")

## 3. Search Tools

Each `@tool` wrapper encapsulates embedding generation and a different level of Cypher graph traversal. The agent calls `vector_search("Apple products")` — the 1024-dimensional embedding stays on the data plane, never passing through the LLM's context window.

The Cypher queries use `WITH node {.*, embedding: null}` to strip the embedding from the return path as well (defense in depth).

| Tool | Graph Traversal |
|------|----------------|
| `vector_search` | None — just chunk text and score |
| `graph_enriched_search` | `FROM_DOCUMENT` + `NEXT_CHUNK` — adds document name and neighboring chunks |
| `entity_enriched_search` | `FROM_DOCUMENT` + `FILED` + `FACES_RISK` + `FROM_CHUNK` — adds companies, products, risk factors |

In [ ]:
@tool
def vector_search(query: str, top_k: int = 3) -> str:
    """Search for semantically similar chunks using vector embeddings.
    Use this for semantic queries about SEC 10-K filing data."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        WITH node {{.*, embedding: null}} AS node, score
        RETURN node.text AS text, score
        ORDER BY score DESC
    """
    result = mcp_client.call_tool_sync(
        tool_use_id="vector-search",
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


@tool
def graph_enriched_search(query: str, top_k: int = 3) -> str:
    """Search for similar chunks enriched with document and neighboring chunk context.
    Returns chunk text, source document, and text from adjacent chunks."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
        OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
        WITH node, doc, score,
             CASE WHEN prev IS NOT NULL THEN prev.text ELSE '' END AS prev_text,
             CASE WHEN next IS NOT NULL THEN next.text ELSE '' END AS next_text
        WITH node {{.*, embedding: null}} AS node, doc, score, prev_text, next_text
        RETURN node.text AS text, score, doc.name AS document,
               prev_text AS previous_chunk, next_text AS next_chunk
        ORDER BY score DESC
    """
    result = mcp_client.call_tool_sync(
        tool_use_id="graph-enriched-search",
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


@tool
def entity_enriched_search(query: str, top_k: int = 3) -> str:
    """Search for similar chunks enriched with companies, products, and risk factors.
    Returns chunk text, source document, and connected entities."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
        OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
        WITH node, doc, score,
             collect(DISTINCT company.name) AS companies,
             collect(DISTINCT risk.name)[0..5] AS risks
        OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
        WITH node, doc, score, companies, risks,
             collect(DISTINCT product.name)[0..5] AS products
        WITH node {{.*, embedding: null}} AS node, doc, score, companies, risks, products
        RETURN node.text AS text, score, doc.name AS document,
               companies, risks, products
        ORDER BY score DESC
    """
    result = mcp_client.call_tool_sync(
        tool_use_id="entity-enriched-search",
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


print("Search tools created:")
print("  - vector_search")
print("  - graph_enriched_search")
print("  - entity_enriched_search")

In [ ]:
VECTOR_ONLY_PROMPT = """You are a retrieval assistant that performs semantic vector search against a Neo4j database containing SEC 10-K filing data.

You have a vector_search tool that finds semantically similar text chunks. Call it with a natural language query and an optional top_k parameter.

For each result, show the similarity score and a preview of the chunk text."""

GRAPH_ENRICHED_PROMPT = """You are a retrieval assistant that performs graph-enriched vector search against a Neo4j database containing SEC 10-K filing data.

You have a graph_enriched_search tool that finds similar chunks and enriches them with the source document name and neighboring chunk text for additional context.

For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Context from neighboring chunks (if available)"""

ENTITY_ENRICHED_PROMPT = """You are a retrieval assistant that performs entity-enriched vector search against a Neo4j database containing SEC 10-K filing data.

You have an entity_enriched_search tool that finds similar chunks and enriches them with connected companies, products, and risk factors from the knowledge graph.

For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Companies, products, and risk factors connected to the chunk"""


async def compare_search(query: str, top_k: int = 3):
    """Run the same query through all three agents and display results."""
    print(f'Query: "{query}"')
    print("=" * 60)

    print("\n--- VECTOR-ONLY RESULTS ---\n")
    vector_agent = Agent(
        model=bedrock_model,
        system_prompt=VECTOR_ONLY_PROMPT,
        tools=[vector_search],
    )
    print(await vector_agent.invoke_async(
        f"Search for: {query}\nUse top_k={top_k}."
    ))

    print("\n\n--- GRAPH-ENRICHED RESULTS ---\n")
    graph_agent = Agent(
        model=bedrock_model,
        system_prompt=GRAPH_ENRICHED_PROMPT,
        tools=[graph_enriched_search],
    )
    print(await graph_agent.invoke_async(
        f"Search for: {query}\nUse top_k={top_k}."
    ))

    print("\n\n--- ENTITY-ENRICHED RESULTS ---\n")
    entity_agent = Agent(
        model=bedrock_model,
        system_prompt=ENTITY_ENRICHED_PROMPT,
        tools=[entity_enriched_search],
    )
    print(await entity_agent.invoke_async(
        f"Search for: {query}\nUse top_k={top_k}."
    ))


print("Compare function ready!")

In [ ]:
await compare_search("What financial metrics indicate company performance?")

## 4. Graph-Enriched Q&A

Now use the entity-enriched agent as a full question-answering pipeline. The agent finds relevant chunks via vector search, gathers graph context, and uses all of that to generate a comprehensive answer.

In [ ]:
QA_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph containing SEC 10-K filing data.

You have an entity_enriched_search tool that searches for relevant chunks enriched with companies, products, and risk factors.

After retrieving results, synthesize a clear answer based on the retrieved context.
Include the companies, products, and risk factors found. Cite the source documents."""


async def ask(query: str, top_k: int = 5):
    """Ask a question using entity-enriched vector search for context."""
    print(f'Question: "{query}"')
    print("-" * 60)

    qa_agent = Agent(
        model=bedrock_model,
        system_prompt=QA_PROMPT,
        tools=[entity_enriched_search],
    )
    response = await qa_agent.invoke_async(
        f"Answer this question using entity-enriched search with top_k={top_k}.\n\n"
        f"Question: {query}"
    )
    print(f"\n{response}")
    return response


print("Q&A function ready!")

In [ ]:
result = await ask("What are the key risk factors mentioned in Apple's 10-K filing?")

In [ ]:
result = await ask("Which companies face cybersecurity-related risks?")

## Summary

You compared three retrieval approaches that show the power of graph-enriched search:

| Approach | What the LLM Receives |
|----------|----------------------|
| **Vector-only** | The matched chunk text and similarity score |
| **Graph-enriched** | The matched chunk + document name + neighboring chunk text |
| **Entity-enriched** | The matched chunk + document + companies, products, risk factors |

Each level of graph traversal adds more context. The entity-enriched approach is the most powerful because it connects unstructured text (chunks) to structured data (entity nodes and their relationships), giving the LLM both the raw filing text and the knowledge graph's understanding of what's in it.

### Embedding Handling

Embeddings are kept off the LLM context window at two levels:

1. **Input path**: `@tool` wrappers encapsulate embedding generation — the agent calls `vector_search("query")`, never seeing the 1024-dim float array
2. **Output path**: Cypher uses `WITH node {.*, embedding: null}` to strip embeddings before returning results, plus explicit property selection in the RETURN clause

This matches the approach used by the neo4j-graphrag library's retrievers (Cypher-level nullification + return property whitelisting).

---

**Next:** [Fulltext and Hybrid Search](03_fulltext_hybrid_search_mcp.ipynb)